# Home Credit Default Risk
## Sprint 3 — Modelagem e Ajuste de Hiperparâmetros

**Dataset:** `application_train.csv` (~307k linhas, 122 colunas)  
**Target:** `TARGET` — 0 = pagou normalmente, 1 = dificuldade de pagamento nas parcelas iniciais  
**Tipo de tarefa:** Classificação binária  

---

## SEÇÃO 1 — Carregamento do Pipeline da Sprint 2

### 1.1 Objetivo

Recarregar os dados processados na Sprint 2 e confirmar que os conjuntos de treino e teste estão prontos para modelagem. **Nenhuma transformação adicional deve ocorrer aqui** — o Pipeline da Sprint 2 deve ser reaplicado exatamente como foi construído, para garantir reprodutibilidade e evitar *data leakage*.

Tópicos a cobrir:

- Carregamento do dataset e reaplicação do Pipeline da Sprint 2 (ou carregamento do dataset processado salvo)
- Confirmação dos shapes de `X_train`, `X_test`, `y_train`, `y_test`
- Verificação de ausência de valores nulos após o pipeline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# ─────────────────────────────────────────────────────────────────────────────
# OPÇÃO A: Recarregar dataset bruto e reaplicar o pipeline da Sprint 2
# ─────────────────────────────────────────────────────────────────────────────
# from sklearn.model_selection import train_test_split
# [cole aqui o código completo do pipeline da Sprint 2]

# ─────────────────────────────────────────────────────────────────────────────
# OPÇÃO B: Carregar dataset processado já salvo (mais rápido)
# ─────────────────────────────────────────────────────────────────────────────
# X_train = pd.read_csv('../data/processed/X_train.csv')
# X_test  = pd.read_csv('../data/processed/X_test.csv')
# y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
# y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

# ─────────────────────────────────────────────────────────────────────────────
# Confirmação dos shapes
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 80)
print("CONFIRMAÇÃO DOS CONJUNTOS")
print("=" * 80)
print(f"\nX_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

print(f"\nDistribuição do target no treino:")
print(y_train.value_counts(normalize=True).round(4))

print(f"\nValores nulos no X_train: {X_train.isnull().sum().sum()}")
print(f"Valores nulos no X_test:  {X_test.isnull().sum().sum()}")

---

## SEÇÃO 2 — Baseline

### 2.1 Definição e justificativa da baseline

A baseline é o modelo mais simples possível para o problema — ela estabelece o piso mínimo de desempenho que qualquer modelo real deve superar. Para um problema de **classificação desbalanceada**, a baseline mais comum é prever sempre a **classe majoritária**.

Tópicos a cobrir:

- Definição da estratégia de baseline (classe majoritária para classificação / média para regressão)
- Cálculo das métricas da baseline com cross-validation (`cv=5`)
- Interpretação: por que a accuracy alta da baseline **não** representa um modelo útil neste problema

In [ ]:
from sklearn.dummy import DummyClassifier  # ou DummyRegressor para regressão
from sklearn.model_selection import cross_val_score

# ─────────────────────────────────────────────────────────────────────────────
# Baseline — DummyClassifier com estratégia 'most_frequent'
# Justificativa: prever sempre a classe majoritária (0) é o modelo mais trivial
# possível. Qualquer modelo com valor preditivo real deve superar esse patamar.
# ─────────────────────────────────────────────────────────────────────────────
baseline = DummyClassifier(strategy='most_frequent', random_state=42)

# Métrica principal do projeto (ajustar conforme o problema)
# Classificação desbalanceada → 'roc_auc' ou 'f1'
METRICA_PRINCIPAL = 'roc_auc'  # AJUSTAR conforme necessidade

scores_baseline = cross_val_score(
    baseline,
    X_train, y_train,
    cv=5,
    scoring=METRICA_PRINCIPAL
)

print("=" * 80)
print("BASELINE — DummyClassifier (classe majoritária)")
print("=" * 80)
print(f"\nMétrica: {METRICA_PRINCIPAL}")
print(f"Scores por fold: {np.round(scores_baseline, 4)}")
print(f"\nMédia:        {scores_baseline.mean():.4f}")
print(f"Desvio Padrão: {scores_baseline.std():.4f}")
print(f"\n📌 Qualquer modelo deve superar {scores_baseline.mean():.4f} para ter valor preditivo real.")

### 2.2 Interpretação da baseline

> **[Escreva aqui sua interpretação]**  
>
> - Por que esse valor de baseline é esperado para este problema?
> - O que significa para o negócio um modelo que se comporta como a baseline?
> - Qual o patamar mínimo aceitável para o projeto?

---

## SEÇÃO 3 — Treinamento e Comparação de Modelos

### 3.1 Seleção e justificativa dos algoritmos

Tópicos a cobrir:

- Seleção de no mínimo 3 algoritmos adequados ao problema
- Justificativa da escolha de cada algoritmo (por que faz sentido para este problema?)
- Para cada modelo: criação de um `Pipeline` completo (pré-processamento + modelo)
- Avaliação com `cross_val_score` (`cv=5`) usando a mesma métrica da baseline
- Tabela comparativa: Modelo | Métrica Média | Desvio Padrão | Observação

**Justificativa dos algoritmos escolhidos:**

| Modelo | Razão da Escolha |
|---|---|
| LogisticRegression | [Justifique] |
| RandomForestClassifier | [Justifique] |
| [Modelo 3] | [Justifique] |
| [Modelo 4 — opcional] | [Justifique] |

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
import time

# ─────────────────────────────────────────────────────────────────────────────
# Identificar tipos de colunas para o ColumnTransformer
# ─────────────────────────────────────────────────────────────────────────────
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"Features numéricas: {len(numeric_features)}")
print(f"Features categóricas: {len(categorical_features)}")

# ─────────────────────────────────────────────────────────────────────────────
# Pré-processador base (adaptar conforme necessidade do projeto)
# ─────────────────────────────────────────────────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), categorical_features)
], remainder='passthrough')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Definição dos Pipelines — um por algoritmo
# ─────────────────────────────────────────────────────────────────────────────
modelos = {
    'LogisticRegression': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'DecisionTree': Pipeline([
        ('preprocessor', preprocessor),
        ('model', DecisionTreeClassifier(random_state=42))
    ]),
    'RandomForest': Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ]),
    # Adicionar mais modelos conforme necessário:
    # 'KNN': Pipeline([...]),
    # 'GradientBoosting': Pipeline([...]),
}

# ─────────────────────────────────────────────────────────────────────────────
# Cross-validation para cada modelo
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 80)
print(f"TREINAMENTO — Cross-Validation (cv=5, métrica={METRICA_PRINCIPAL})")
print("=" * 80)

resultados_cv = []

for nome, pipeline in modelos.items():
    print(f"\nTreinando: {nome}...")
    inicio = time.time()

    scores = cross_val_score(
        pipeline,
        X_train, y_train,
        cv=5,
        scoring=METRICA_PRINCIPAL,
        n_jobs=-1
    )

    tempo = time.time() - inicio

    resultados_cv.append({
        'Modelo': nome,
        'Média': scores.mean(),
        'Desvio Padrão': scores.std(),
        'Tempo (s)': round(tempo, 1)
    })

    print(f"  {METRICA_PRINCIPAL}: {scores.mean():.4f} ± {scores.std():.4f}  ({tempo:.1f}s)")

# ─────────────────────────────────────────────────────────────────────────────
# Tabela comparativa
# ─────────────────────────────────────────────────────────────────────────────
df_resultados = pd.DataFrame(resultados_cv).sort_values('Média', ascending=False).reset_index(drop=True)

# Adicionar baseline como referência
baseline_row = pd.DataFrame([{
    'Modelo': '⚑ Baseline (DummyClassifier)',
    'Média': scores_baseline.mean(),
    'Desvio Padrão': scores_baseline.std(),
    'Tempo (s)': '-'
}])
df_resultados = pd.concat([df_resultados, baseline_row], ignore_index=True)

print("\n" + "=" * 80)
print("TABELA COMPARATIVA — Modelos vs. Baseline")
print("=" * 80)
print(df_resultados.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Visualização comparativa
# ─────────────────────────────────────────────────────────────────────────────
modelos_plot = df_resultados[df_resultados['Modelo'] != '⚑ Baseline (DummyClassifier)'].copy()
baseline_val = scores_baseline.mean()

fig, ax = plt.subplots(figsize=(10, 5))
cores = ['#2ecc71' if m > baseline_val else '#e74c3c'
         for m in modelos_plot['Média']]

bars = ax.barh(
    modelos_plot['Modelo'],
    modelos_plot['Média'],
    xerr=modelos_plot['Desvio Padrão'],
    color=cores,
    edgecolor='white',
    linewidth=1.2,
    capsize=5
)

ax.axvline(baseline_val, color='#e67e22', linewidth=2,
           linestyle='--', label=f'Baseline ({baseline_val:.4f})')

for bar, row in zip(bars, modelos_plot.itertuples()):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{row.Média:.4f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel(METRICA_PRINCIPAL.upper(), fontsize=12)
ax.set_title(f'Comparação de Modelos — {METRICA_PRINCIPAL.upper()} (cv=5)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### 3.2 Análise dos resultados

> **[Escreva aqui sua análise]**  
>
> - Qual modelo apresentou o melhor desempenho? Houve surpresas?
> - Como a estabilidade (desvio padrão) influencia a escolha?
> - Algum modelo ficou abaixo da baseline? O que isso indica?
> - Quais os 2 modelos mais promissores para ajuste de hiperparâmetros?

---

## SEÇÃO 4 — Ajuste de Hiperparâmetros

### 4.1 Seleção dos modelos e estratégia de busca

Os 2 modelos com melhor desempenho na Seção 3 serão submetidos a ajuste de hiperparâmetros.

Tópicos a cobrir:

- Identificação dos 2 melhores modelos da Seção 3
- Justificativa da estratégia de busca escolhida (`GridSearchCV` vs. `RandomizedSearchCV`)
- Documentação do espaço de busca (quais hiperparâmetros e por quê)
- Comparação: modelo padrão vs. modelo ajustado

**Estratégia de busca:**

| Modelo | Estratégia | Justificativa |
|---|---|---|
| [Modelo 1] | GridSearchCV / RandomizedSearchCV | [Justifique] |
| [Modelo 2] | GridSearchCV / RandomizedSearchCV | [Justifique] |

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# ─────────────────────────────────────────────────────────────────────────────
# MODELO 1 — [Nome do modelo]
# Estratégia: [GridSearchCV ou RandomizedSearchCV]
# Justificativa: [explicar a escolha]
# ─────────────────────────────────────────────────────────────────────────────

# Espaço de busca — ajustar conforme o modelo escolhido
param_grid_modelo1 = {
    # Exemplo para RandomForest:
    # 'model__n_estimators': [100, 200, 300],
    # 'model__max_depth': [None, 10, 20, 30],
    # 'model__min_samples_split': [2, 5, 10],
    # 'model__min_samples_leaf': [1, 2, 4],
}

pipeline_modelo1 = modelos['RandomForest']  # ALTERAR para o modelo escolhido

search_modelo1 = GridSearchCV(
    pipeline_modelo1,
    param_grid=param_grid_modelo1,
    cv=5,
    scoring=METRICA_PRINCIPAL,
    n_jobs=-1,
    verbose=1,
    refit=True
)

print("=" * 80)
print("AJUSTE DE HIPERPARÂMETROS — Modelo 1: [nome]")
print("=" * 80)
print(f"Espaço de busca: {param_grid_modelo1}")
print()

search_modelo1.fit(X_train, y_train)

print(f"\nMelhores hiperparâmetros:")
for param, val in search_modelo1.best_params_.items():
    print(f"  {param}: {val}")
print(f"\nMelhor score (cv=5): {search_modelo1.best_score_:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MODELO 2 — [Nome do modelo]
# Estratégia: [GridSearchCV ou RandomizedSearchCV]
# Justificativa: [explicar a escolha]
# ─────────────────────────────────────────────────────────────────────────────

param_grid_modelo2 = {
    # Exemplo para LogisticRegression:
    # 'model__C': [0.001, 0.01, 0.1, 1, 10, 100],
    # 'model__penalty': ['l1', 'l2'],
    # 'model__solver': ['liblinear', 'saga'],
}

pipeline_modelo2 = modelos['LogisticRegression']  # ALTERAR para o modelo escolhido

search_modelo2 = RandomizedSearchCV(
    pipeline_modelo2,
    param_distributions=param_grid_modelo2,
    n_iter=20,
    cv=5,
    scoring=METRICA_PRINCIPAL,
    n_jobs=-1,
    verbose=1,
    random_state=42,
    refit=True
)

print("=" * 80)
print("AJUSTE DE HIPERPARÂMETROS — Modelo 2: [nome]")
print("=" * 80)
print(f"Espaço de busca: {param_grid_modelo2}")
print()

search_modelo2.fit(X_train, y_train)

print(f"\nMelhores hiperparâmetros:")
for param, val in search_modelo2.best_params_.items():
    print(f"  {param}: {val}")
print(f"\nMelhor score (cv=5): {search_modelo2.best_score_:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Comparação: padrão vs. ajustado
# ─────────────────────────────────────────────────────────────────────────────
nome_modelo1 = 'RandomForest'   # ALTERAR
nome_modelo2 = 'LogisticRegression'  # ALTERAR

score_padrao_m1 = df_resultados[df_resultados['Modelo'] == nome_modelo1]['Média'].values[0]
score_padrao_m2 = df_resultados[df_resultados['Modelo'] == nome_modelo2]['Média'].values[0]

comparacao = pd.DataFrame([
    {'Modelo': nome_modelo1, 'Padrão': score_padrao_m1,
     'Ajustado': search_modelo1.best_score_,
     'Δ Melhora': search_modelo1.best_score_ - score_padrao_m1},
    {'Modelo': nome_modelo2, 'Padrão': score_padrao_m2,
     'Ajustado': search_modelo2.best_score_,
     'Δ Melhora': search_modelo2.best_score_ - score_padrao_m2},
])

print("=" * 80)
print("COMPARAÇÃO: Modelo Padrão vs. Modelo Ajustado")
print("=" * 80)
print(comparacao.to_string(index=False))

### 4.2 Análise do ajuste de hiperparâmetros

> **[Escreva aqui sua análise]**  
>
> - Houve melhora significativa após o ajuste? Em qual modelo?
> - Os melhores hiperparâmetros fazem sentido intuitivo? Por quê?
> - O ajuste valeu o custo computacional?

---

## SEÇÃO 5 — Escolha do Modelo Final

### 5.1 Critérios de decisão

Com base nos resultados das Seções 3 e 4, selecionar **um** modelo final, justificando a escolha considerando os seguintes critérios:

| Critério | Peso para o Projeto | Modelo Escolhido |
|---|---|---|
| **Desempenho** (métrica principal) | [alto / médio / baixo] | |
| **Estabilidade** (desvio padrão no cv) | [alto / médio / baixo] | |
| **Interpretabilidade** | [alto / médio / baixo] | |
| **Complexidade computacional** | [alto / médio / baixo] | |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Definição do modelo final
# ─────────────────────────────────────────────────────────────────────────────

# Selecionar o search com o melhor score entre os dois ajustados
if search_modelo1.best_score_ >= search_modelo2.best_score_:
    melhor_search = search_modelo1
    nome_modelo_final = nome_modelo1
else:
    melhor_search = search_modelo2
    nome_modelo_final = nome_modelo2

modelo_final = melhor_search.best_estimator_

print("=" * 80)
print("MODELO FINAL SELECIONADO")
print("=" * 80)
print(f"\nModelo: {nome_modelo_final}")
print(f"Score cross-validation (cv=5): {melhor_search.best_score_:.4f}")
print(f"\nHiperparâmetros finais:")
for param, val in melhor_search.best_params_.items():
    print(f"  {param}: {val}")

### 5.2 Justificativa narrativa da escolha

> **[Escreva aqui sua justificativa completa]**  
>
> Deve cobrir:
> - Por que este modelo foi escolhido em detrimento dos demais?
> - Como o desempenho, a estabilidade, a interpretabilidade e a complexidade computacional pesaram na decisão?
> - Existe algum risco ou limitação relevante no modelo escolhido?

---

## SEÇÃO 6 — Avaliação Final no Conjunto de Teste

### ⚠️ ATENÇÃO

> **O conjunto de teste (`X_test`, `y_test`) é usado UMA ÚNICA VEZ, aqui.** Toda a comparação e seleção de modelos foi realizada apenas com o conjunto de treino via cross-validation. Usar o teste mais de uma vez invalida a avaliação e introduz *data leakage*.

Tópicos a cobrir:

- Predição do modelo final no `X_test`
- Cálculo de todas as métricas relevantes para o tipo de problema
- Gráficos obrigatórios (Matriz de Confusão, Curva ROC/AUC para classificação)
- Comparação do score de teste com o score do cross-validation

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# ─────────────────────────────────────────────────────────────────────────────
# Predição no conjunto de teste — UMA ÚNICA VEZ
# ─────────────────────────────────────────────────────────────────────────────
y_pred = modelo_final.predict(X_test)
y_pred_proba = modelo_final.predict_proba(X_test)[:, 1]  # remover se o modelo não suportar

print("=" * 80)
print("AVALIAÇÃO FINAL — Conjunto de Teste")
print("=" * 80)

# ─────────────────────────────────────────────────────────────────────────────
# Métricas principais
# ─────────────────────────────────────────────────────────────────────────────
acuracia  = accuracy_score(y_test, y_pred)
precisao  = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)
auc       = roc_auc_score(y_test, y_pred_proba)

print(f"\n{'Métrica':<25} {'Valor':>10}")
print("-" * 36)
print(f"{'Acurácia':<25} {acuracia:>10.4f}")
print(f"{'Precision':<25} {precisao:>10.4f}")
print(f"{'Recall':<25} {recall:>10.4f}")
print(f"{'F1-Score':<25} {f1:>10.4f}")
print(f"{'ROC-AUC':<25} {auc:>10.4f}")

print(f"\n{'Score cross-validation (cv=5)':<35} {melhor_search.best_score_:.4f}")
print(f"{'Score no conjunto de teste':<35} {auc:.4f}")
print(f"{'Diferença':<35} {abs(auc - melhor_search.best_score_):.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Relatório por classe
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 80)
print("RELATÓRIO POR CLASSE (classification_report)")
print("=" * 80)
print(classification_report(y_test, y_pred,
                             target_names=['0 — Pagou normalmente', '1 — Dificuldade de pagamento']))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Visualizações: Matriz de Confusão + Curva ROC
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Matriz de Confusão ---
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Previsto 0', 'Previsto 1'],
    yticklabels=['Real 0', 'Real 1'],
    ax=axes[0], linewidths=0.5
)
axes[0].set_title('Matriz de Confusão', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Classe Real', fontsize=12)
axes[0].set_xlabel('Classe Prevista', fontsize=12)

# --- Curva ROC ---
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, color='#3498db', linewidth=2,
             label=f'{nome_modelo_final} (AUC = {auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Baseline aleatória')
axes[1].set_xlabel('Taxa de Falsos Positivos (FPR)', fontsize=12)
axes[1].set_ylabel('Taxa de Verdadeiros Positivos (TPR)', fontsize=12)
axes[1].set_title('Curva ROC', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Avaliação Final — {nome_modelo_final}', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 6.1 Análise da avaliação final

> **[Escreva aqui sua análise]**  
>
> - O score no teste foi próximo do score no cross-validation? O modelo generalizou bem?
> - Se houver grande diferença, quais são as possíveis causas (overfitting, distribuição dos dados)?
> - O que a Matriz de Confusão revela sobre os erros do modelo? Falsos negativos vs. falsos positivos — qual tem maior custo para o negócio?
> - O AUC obtido é satisfatório para o problema de crédito?

---

## SEÇÃO 7 — Persistência do Modelo

### 7.1 Salvando o Pipeline final

O pipeline final (pré-processamento + modelo com os melhores hiperparâmetros) é salvo em disco para uso em produção ou nas próximas sprints, sem necessidade de re-treinar.

In [ ]:
import joblib
import os

# ─────────────────────────────────────────────────────────────────────────────
# Salvar o pipeline final
# ─────────────────────────────────────────────────────────────────────────────
os.makedirs('../models', exist_ok=True)
caminho_modelo = '../models/modelo_projeto.pkl'

joblib.dump(modelo_final, caminho_modelo)
print(f"✓ Modelo salvo em: {caminho_modelo}")

# ─────────────────────────────────────────────────────────────────────────────
# Verificação: carregar e confirmar que produz as mesmas previsões
# ─────────────────────────────────────────────────────────────────────────────
modelo_carregado = joblib.load(caminho_modelo)
y_pred_verificacao = modelo_carregado.predict(X_test)

verificacao_ok = np.array_equal(y_pred, y_pred_verificacao)
print(f"\n✓ Verificação de integridade: previsões idênticas = {verificacao_ok}")

print("\n" + "=" * 80)
print("RESUMO FINAL DA SPRINT 3")
print("=" * 80)
print(f"  Modelo final:            {nome_modelo_final}")
print(f"  Score cv (treino):       {melhor_search.best_score_:.4f}")
print(f"  ROC-AUC (teste):         {auc:.4f}")
print(f"  F1-Score (teste):        {f1:.4f}")
print(f"  Modelo salvo em:         {caminho_modelo}")
print(f"  Integridade verificada:  {verificacao_ok}")

### 7.2 Justificativa da persistência

> **[Escreva aqui sua justificativa]**  
>
> - Por que salvar o pipeline completo (pré-processamento + modelo) e não apenas o modelo?
> - Quais dependências são necessárias para carregar e usar o modelo em outro ambiente?